# BigQuery Catalog Mapper — LangGraph + Gemini 2.5 (Vertex AI)

An **autonomous** data-modeling agent that explores a large BigQuery warehouse (hundreds of
tables) on its own and builds a trustworthy catalog: each table's **purpose & grain**, its
**business rules**, its **key columns**, and the **principal join attributes** that link tables —
exported as `data_catalog.json`, `data_dictionary.md`, `join_candidates.csv` and a
`WAREHOUSE_OVERVIEW.md` memo.

It is **not a chat tool**. The objective is autonomous coverage, so the harness — not the model's
patience — drives it: the unit of work is **one table**, and mapping the warehouse is a **graph
walk** over a worklist.

This is the reliability architecture of
[`code_assistant/claude_code_from_scratch_v2.ipynb`](../code_assistant/claude_code_from_scratch_v2.ipynb),
re-expressed in **LangGraph** (for maintainability) on **Gemini 2.5 via Vertex AI**. It mirrors the
house LangGraph idioms in `capital_allocation.ipynb` (`ChatVertexAI`, `@tool` + `bind_tools`,
`StateGraph`/`ToolNode`, `MemorySaver`).

### v2 techniques → LangGraph form

| v2 technique | here |
|---|---|
| problem router | **table-type classifier** (`with_structured_output`) → cost-aware *profiling depth* (BigQuery bytes) |
| DAG-driven coverage | an **orchestrator `StateGraph`** that walks a per-table worklist: pick → profile → verify → retry/next → synthesize |
| bounded context / subagent | a fresh **per-table ReAct profiler subgraph** (isolated messages) |
| spec layer (pytest) | **`verify_table_entry`**: required fields + **re-running `bq_join_overlap`** for high-confidence joins (independent recomputation) |
| durable state | `MemorySaver` checkpointer (per `thread_id`) + the on-disk catalog JSON → resumable |

### Cost & safety posture
Free metadata does the heavy lifting (schemas, row counts, `INFORMATION_SCHEMA`). Every **data**
read is read-only `SELECT`/`WITH`, **dry-run first**, refused above `BQ_MAX_BYTES_BILLED`, and run
with `maximum_bytes_billed` set. The table-type router gates how many bytes each table earns. Point
it at a warehouse only with least-privilege credentials; queries and model calls bill your project.

### Setup
```bash
pip install langgraph langchain-google-vertexai langchain-core google-cloud-bigquery
gcloud auth application-default login        # ADC drives both Vertex AI and BigQuery
export GOOGLE_CLOUD_PROJECT=your-gcp-project
export GOOGLE_CLOUD_LOCATION=us-central1
# optional: export BQ_DATASETS=ds1,ds2   CATALOG_MAX_TABLES=12
```

## Install (uncomment on first run)

In [ ]:
# Optional: install dependencies (uncomment on first run)
# %pip install -q langgraph langchain-google-vertexai langchain-core google-cloud-bigquery

In [ ]:
"""
Imports + configuration. All knobs live here.

Backend: Gemini 2.5 via Vertex AI (ChatVertexAI). Data: read-only BigQuery. Both authenticate
through Application Default Credentials in your GCP environment.
"""
from __future__ import annotations

import json
import logging
import os
import re
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Literal

# --- GCP / model -------------------------------------------------------------
PROJECT  = os.environ.get("GOOGLE_CLOUD_PROJECT", "")               # set this
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")   # Vertex region
MODEL    = os.environ.get("CATALOG_MODEL", "gemini-2.5-pro")
FAST_MODEL = os.environ.get("CATALOG_FAST_MODEL", "gemini-2.5-flash")  # classifier/profiler
THINKING_BUDGET = int(os.environ.get("CATALOG_THINKING_BUDGET", "-1"))  # -1 = auto

# --- BigQuery ----------------------------------------------------------------
BQ_LOCATION         = os.environ.get("BQ_LOCATION", "US")
# Comma-separated allow-list to scope the work; empty = every dataset in the project.
BQ_DATASETS         = [d.strip() for d in os.environ.get("BQ_DATASETS", "").split(",") if d.strip()]
BQ_MAX_BYTES_BILLED = int(os.environ.get("BQ_MAX_BYTES_BILLED", str(1 << 30)))  # 1 GiB hard cap
BQ_MAX_RESULT_ROWS  = int(os.environ.get("BQ_MAX_RESULT_ROWS", "200"))
BQ_SAMPLE_ROWS      = int(os.environ.get("BQ_SAMPLE_ROWS", "8"))
BQ_MAX_DATASETS_SCAN = int(os.environ.get("BQ_MAX_DATASETS_SCAN", "60"))

# --- outputs / durable state -------------------------------------------------
WORKDIR = Path(os.environ.get("CATALOG_WORKDIR", str(Path.cwd() / "catalog_out"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
CATALOG_FILE = WORKDIR / "data_catalog.json"     # the live, resumable catalog

# --- autonomous-mapper knobs -------------------------------------------------
PROFILER_MAX_STEPS  = int(os.environ.get("CATALOG_PROFILER_MAX_STEPS", "24"))  # ReAct steps / table
MAP_MAX_ATTEMPTS    = int(os.environ.get("CATALOG_MAX_ATTEMPTS", "2"))         # profile+verify retries
# Re-confirm a high-confidence join only if value overlap >= this (independent recomputation).
VERIFY_OVERLAP_THRESHOLD = float(os.environ.get("CATALOG_VERIFY_OVERLAP_THRESHOLD", "0.50"))

logging.basicConfig(level=os.environ.get("AGENT_LOG_LEVEL", "INFO").upper(),
                    format="[%(levelname)s] %(name)s | %(message)s")
log = logging.getLogger("catalog")
log.info("model=%s fast=%s vertex=%s bq_location=%s datasets=%s",
         MODEL, FAST_MODEL, LOCATION, BQ_LOCATION, BQ_DATASETS or "(all)")

## Read-only BigQuery tools

Built on `google-cloud-bigquery` so we own the safety posture. Free metadata tools orient;
data-reading tools are dry-run first, byte-capped, and row-capped. Each `@tool` wraps a plain `_fn`
so the autonomous orchestrator can call the same logic directly, without a model round-trip.

In [ ]:
"""
Read-only BigQuery layer with cost guards, exposed as LangChain @tools.

Free metadata tools do the orientation; data-reading tools are DRY-RUN first, refused above
BQ_MAX_BYTES_BILLED, and run with maximum_bytes_billed set. Each @tool wraps a plain `_fn` so the
autonomous orchestrator can call the same logic directly (without going through the model).
"""
from google.cloud import bigquery
from langchain_core.tools import tool

_bq = None

def bq_client() -> "bigquery.Client":
    global _bq
    if _bq is None:
        _bq = bigquery.Client(project=PROJECT or None, location=BQ_LOCATION)
    return _bq


_READONLY_RE  = re.compile(r"^\s*(SELECT|WITH)\b", re.I)
_FORBIDDEN_RE = re.compile(r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|CREATE|ALTER|TRUNCATE|GRANT|"
                           r"REVOKE|CALL|EXPORT|LOAD)\b", re.I)
_IDENT_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
_LIKE_RE  = re.compile(r"^[A-Za-z0-9_%]+$")


def _safe_ident(s: str) -> str:
    if not _IDENT_RE.match(s or ""):
        raise ValueError(f"unsafe identifier: {s!r}")
    return s


def _split_table(table: str):
    """Accept 'dataset.table' or 'project.dataset.table' -> (project, dataset, table)."""
    parts = table.replace("`", "").strip().split(".")
    if len(parts) == 3:
        proj, ds, tbl = parts
    elif len(parts) == 2:
        proj, ds, tbl = (PROJECT or bq_client().project), parts[0], parts[1]
    else:
        raise ValueError(f"table must be 'dataset.table' or 'project.dataset.table', got {table!r}")
    return proj, _safe_ident(ds), _safe_ident(tbl)


def _guarded_query(sql: str, row_limit: int = None):
    """Validate (read-only), dry-run for cost, then execute capped. Returns (rows, error)."""
    if not _READONLY_RE.match(sql or "") or _FORBIDDEN_RE.search(sql or ""):
        return None, "Refused: only read-only SELECT/WITH queries are allowed."
    try:
        dry = bq_client().query(
            sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        scanned = dry.total_bytes_processed or 0
    except Exception as e:
        return None, f"Error: dry-run failed: {e}"
    if scanned > BQ_MAX_BYTES_BILLED:
        return None, (f"Refused: query would scan {scanned/1e9:.2f} GB, over the "
                      f"{BQ_MAX_BYTES_BILLED/1e9:.2f} GB cap. Add filters / partition / LIMIT.")
    try:
        job = bq_client().query(
            sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=BQ_MAX_BYTES_BILLED))
        rows = [dict(r) for r in job.result(max_results=row_limit or BQ_MAX_RESULT_ROWS)]
        return rows, None
    except Exception as e:
        return None, f"Error: {e}"


# ---- plain helpers (used by the orchestrator) -------------------------------
def _list_datasets() -> List[str]:
    ids = [d.dataset_id for d in bq_client().list_datasets()]
    return [d for d in ids if d in set(BQ_DATASETS)] if BQ_DATASETS else ids


def _list_tables(dataset: str) -> List[str]:
    return [t.table_id for t in bq_client().list_tables(f"{bq_client().project}.{_safe_ident(dataset)}")]


def _table_schema(table: str) -> str:
    proj, ds, tbl = _split_table(table)
    t = bq_client().get_table(f"{proj}.{ds}.{tbl}")   # free metadata
    cols = [{"name": f.name, "type": f.field_type, "mode": f.mode,
             "description": f.description or ""} for f in t.schema]
    return json.dumps({
        "table": f"{ds}.{tbl}", "rows": t.num_rows, "size_mb": round((t.num_bytes or 0) / 1e6, 2),
        "partitioning": (t.time_partitioning.field if t.time_partitioning else None),
        "clustering": t.clustering_fields, "description": t.description or "", "columns": cols,
    }, default=str)


def _join_overlap_stats(left_table: str, left_col: str, right_table: str, right_col: str,
                        sample: int = 100_000):
    """Numeric value-overlap (capped DISTINCT sample). Returns a dict or None on refusal/error.
    This is what the Verifier re-runs to objectively confirm a high-confidence join."""
    try:
        lp, lds, lt = _split_table(left_table)
        rp, rds, rt = _split_table(right_table)
        lc, rc = _safe_ident(left_col), _safe_ident(right_col)
    except Exception as e:
        log.warning("[overlap] bad args: %s", e)
        return None
    sample = max(1000, min(int(sample), 2_000_000))
    sql = (f"WITH l AS (SELECT DISTINCT CAST(`{lc}` AS STRING) v FROM `{lp}`.`{lds}`.`{lt}` "
           f"WHERE `{lc}` IS NOT NULL LIMIT {sample}), "
           f"r AS (SELECT DISTINCT CAST(`{rc}` AS STRING) v FROM `{rp}`.`{rds}`.`{rt}` "
           f"WHERE `{rc}` IS NOT NULL LIMIT {sample}) "
           f"SELECT (SELECT COUNT(*) FROM l) l_distinct, (SELECT COUNT(*) FROM r) r_distinct, "
           f"(SELECT COUNT(*) FROM l JOIN r USING (v)) overlap")
    rows, err = _guarded_query(sql, row_limit=1)
    if rows is None:
        return None
    r = rows[0]
    ld, rd, ov = int(r["l_distinct"]), int(r["r_distinct"]), int(r["overlap"])
    return {"l_distinct": ld, "r_distinct": rd, "overlap": ov,
            "left_cov": (ov / ld) if ld else 0.0, "right_cov": (ov / rd) if rd else 0.0}


# ---- @tools (bound to the profiler agent) -----------------------------------
@tool
def bq_list_datasets() -> str:
    """List dataset IDs in the BigQuery project (respects the BQ_DATASETS allow-list). Free."""
    try:
        return json.dumps(_list_datasets())
    except Exception as e:
        return f"Error: {e}"


@tool
def bq_list_tables(dataset: str) -> str:
    """List table IDs in a dataset. Free. Args: dataset."""
    try:
        return json.dumps(_list_tables(dataset))
    except Exception as e:
        return f"Error: {e}"


@tool
def bq_table_schema(table: str) -> str:
    """Schema (columns/types/modes/descriptions), row count, size, partitioning and clustering for
    one table. FREE metadata -- your primary tool to read grain and find join keys. Args: table as
    'dataset.table' or 'project.dataset.table'."""
    try:
        return _table_schema(table)
    except Exception as e:
        return f"Error: {e}"


@tool
def bq_sample(table: str, n: int = BQ_SAMPLE_ROWS) -> str:
    """Return a few example rows (SELECT * LIMIT n). Dry-run + byte-capped. Args: table, n (<=50)."""
    try:
        proj, ds, tbl = _split_table(table)
    except Exception as e:
        return f"Error: {e}"
    n = max(1, min(int(n), 50))
    rows, err = _guarded_query(f"SELECT * FROM `{proj}`.`{ds}`.`{tbl}` LIMIT {n}", row_limit=n)
    return err or json.dumps(rows, default=str)


@tool
def bq_query(sql: str) -> str:
    """Run a READ-ONLY SELECT/WITH query (dry-run first, byte-capped, row-capped). Use COUNTIF,
    APPROX_COUNT_DISTINCT, partition filters and LIMIT to stay cheap. Good for profiling (null %,
    distinct counts, value distributions) and checking business rules. Args: sql. Returns JSON rows."""
    rows, err = _guarded_query(sql)
    return err or json.dumps(rows, default=str)


@tool
def bq_columns(name_like: str, dataset: str = "") -> str:
    """Find columns across datasets whose NAME matches a SQL LIKE pattern (e.g. '%_id', 'account%')
    -- the core join-key discovery tool. Cheap INFORMATION_SCHEMA query. Args: name_like, dataset
    (one dataset, or '' for all in scope)."""
    if not _LIKE_RE.match(name_like or ""):
        return "Error: name_like may contain only letters, digits, underscore and % (LIKE)."
    try:
        datasets = [_safe_ident(dataset)] if dataset else (BQ_DATASETS or _list_datasets())
        datasets = datasets[:BQ_MAX_DATASETS_SCAN]
    except Exception as e:
        return f"Error: {e}"
    proj = bq_client().project
    hits = []
    for ds in datasets:
        rows, err = _guarded_query(
            f"SELECT table_name, column_name, data_type FROM `{proj}`.`{ds}`.INFORMATION_SCHEMA.COLUMNS "
            f"WHERE column_name LIKE '{name_like}' ORDER BY column_name, table_name", row_limit=1000)
        if rows:
            hits += [f"{ds}.{r['table_name']}.{r['column_name']} ({r['data_type']})" for r in rows]
    return json.dumps(hits[:400]) if hits else f"No columns matching {name_like!r}."


@tool
def bq_join_overlap(left_table: str, left_col: str, right_table: str, right_col: str,
                    sample: int = 100_000) -> str:
    """Evidence for a join/FK: measure how much two columns' value sets overlap (capped DISTINCT
    sample). High left->right coverage implies left is a FK into right. Dry-run + byte-capped.
    Args: left_table, left_col, right_table, right_col, sample."""
    stats = _join_overlap_stats(left_table, left_col, right_table, right_col, sample)
    if stats is None:
        return "Error/refused: overlap check could not run (bad args or over byte cap)."
    return json.dumps({**stats, "left_cov": round(stats["left_cov"], 4),
                       "right_cov": round(stats["right_cov"], 4)})


log.info("BigQuery tools ready (metadata + capped data reads + join-overlap evidence).")

## Catalog store + tools

The catalog is a durable, resumable JSON document on disk. The profiler agent writes into it with
`catalog_set_table` / `catalog_add_join`; the Verifier and the synthesis step read it back.

In [ ]:
"""
Catalog store + tools. The catalog is a JSON document on disk (CATALOG_FILE) -- durable and
resumable, so a multi-hour mapping run can be paused/resumed. The profiler agent writes into it
with catalog_set_table / catalog_add_join; the Verifier and the synthesis step read it back.

Shape:
  {"tables": {"dataset.table": {purpose, grain, row_count, key_columns[], business_rules[], notes}},
   "joins":  [{left, left_col, right, right_col, confidence, basis, evidence}]}
"""
from langchain_core.tools import tool

JOIN_CONFIDENCE = ("high", "medium", "low")
JOIN_BASIS = ("name+type", "fk-naming", "pk-unique", "value-overlap", "manual")


def _empty_catalog():
    return {"tables": {}, "joins": []}


def _load_catalog():
    if not CATALOG_FILE.exists():
        return _empty_catalog()
    try:
        c = json.loads(CATALOG_FILE.read_text(encoding="utf-8"))
        c.setdefault("tables", {}); c.setdefault("joins", [])
        return c
    except (json.JSONDecodeError, OSError):
        return _empty_catalog()


def _save_catalog(c):
    CATALOG_FILE.write_text(json.dumps(c, indent=2), encoding="utf-8")


def new_catalog():
    """Start a fresh catalog (deletes CATALOG_FILE)."""
    try:
        CATALOG_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log.info("catalog reset")


def _as_list(v):
    if v is None:
        return []
    return [v] if isinstance(v, str) else list(v)


@tool
def catalog_set_table(table: str, purpose: str = "", grain: str = "", row_count: int = None,
                      key_columns: list = None, business_rules: list = None, notes: str = "") -> str:
    """Record/merge what you learned about a table. Args: table ('dataset.table'), purpose,
    grain (what ONE row represents), row_count, key_columns (PK/identifier columns),
    business_rules (observed constraints/semantics), notes. Only record evidence-backed facts."""
    c = _load_catalog()
    t = c["tables"].get(table, {"purpose": "", "grain": "", "row_count": None,
                                "key_columns": [], "business_rules": [], "notes": ""})
    if purpose: t["purpose"] = purpose
    if grain: t["grain"] = grain
    if row_count is not None: t["row_count"] = row_count
    for k in _as_list(key_columns):
        if k not in t["key_columns"]: t["key_columns"].append(k)
    for br in _as_list(business_rules):
        if br not in t["business_rules"]: t["business_rules"].append(br)
    if notes: t["notes"] = (t.get("notes", "") + ("\n" if t.get("notes") else "") + notes)
    c["tables"][table] = t
    _save_catalog(c)
    return f"catalogued {table}: purpose={'set' if t['purpose'] else '-'} grain={'set' if t['grain'] else '-'} keys={t['key_columns']}"


@tool
def catalog_add_join(left: str, left_col: str, right: str, right_col: str,
                     confidence: str = "medium", basis: str = "name+type", evidence: str = "") -> str:
    """Record a candidate join between two tables. confidence=high|medium|low;
    basis=name+type|fk-naming|pk-unique|value-overlap|manual. Cite evidence (esp. measured overlap
    for value-overlap). Args: left, left_col, right, right_col, confidence, basis, evidence."""
    if confidence not in JOIN_CONFIDENCE:
        return f"Error: confidence must be one of {JOIN_CONFIDENCE}"
    if basis not in JOIN_BASIS:
        return f"Error: basis must be one of {JOIN_BASIS}"
    c = _load_catalog()
    for j in c["joins"]:
        if (j["left"], j["left_col"], j["right"], j["right_col"]) == (left, left_col, right, right_col):
            j.update({"confidence": confidence, "basis": basis, "evidence": evidence or j.get("evidence", "")})
            _save_catalog(c)
            return f"join updated: {left}.{left_col}={right}.{right_col} [{confidence}/{basis}]"
    c["joins"].append({"left": left, "left_col": left_col, "right": right, "right_col": right_col,
                       "confidence": confidence, "basis": basis, "evidence": evidence})
    _save_catalog(c)
    return f"join added: {left}.{left_col}={right}.{right_col} [{confidence}/{basis}]"


def run_catalog_summary() -> str:
    c = _load_catalog()
    tables, joins = c["tables"], c["joins"]
    described = sum(1 for t in tables.values() if t.get("purpose"))
    ruled = sum(len(t.get("business_rules", [])) for t in tables.values())
    by_conf = {}
    deg = {}
    for j in joins:
        by_conf[j["confidence"]] = by_conf.get(j["confidence"], 0) + 1
        deg[j["left"]] = deg.get(j["left"], 0) + 1
        deg[j["right"]] = deg.get(j["right"], 0) + 1
    hubs = sorted(deg.items(), key=lambda kv: -kv[1])[:5]
    return "\n".join([
        f"tables catalogued: {len(tables)} ({described} with a purpose)",
        f"business rules captured: {ruled}",
        f"join candidates: {len(joins)} by confidence {by_conf}",
        f"most-connected tables (hubs): {hubs}",
    ])


def _to_dictionary(c):
    L = ["# Data Dictionary", "", f"_{len(c['tables'])} tables catalogued._", ""]
    for name in sorted(c["tables"]):
        t = c["tables"][name]
        L.append(f"## `{name}`")
        if t.get("row_count") is not None: L.append(f"- **Rows:** {t['row_count']:,}")
        if t.get("purpose"): L.append(f"- **Purpose:** {t['purpose']}")
        if t.get("grain"): L.append(f"- **Grain:** {t['grain']}")
        if t.get("key_columns"): L.append(f"- **Key columns:** {', '.join('`'+k+'`' for k in t['key_columns'])}")
        if t.get("business_rules"):
            L.append("- **Business rules:**"); L += [f"    - {br}" for br in t["business_rules"]]
        if t.get("notes"): L.append(f"- **Notes:** {t['notes']}")
        L.append("")
    return "\n".join(L)


def _to_join_csv(c):
    order = {"high": 0, "medium": 1, "low": 2}
    rows = sorted(c["joins"], key=lambda j: (order.get(j["confidence"], 9), j["left"]))
    out = ["left_table,left_col,right_table,right_col,confidence,basis,evidence"]
    for j in rows:
        ev = str(j.get("evidence", "")).replace('"', "'").replace("\n", " ")
        out.append(f'"{j["left"]}","{j["left_col"]}","{j["right"]}","{j["right_col"]}",'
                   f'{j["confidence"]},{j["basis"]},"{ev}"')
    return "\n".join(out)


def run_catalog_export(fmt: str = "all") -> List[str]:
    c = _load_catalog()
    fmt = (fmt or "all").lower()
    written = []
    if fmt in ("json", "all"):
        CATALOG_FILE.write_text(json.dumps(c, indent=2), encoding="utf-8"); written.append(str(CATALOG_FILE))
    if fmt in ("dictionary", "all"):
        p = WORKDIR / "data_dictionary.md"; p.write_text(_to_dictionary(c), encoding="utf-8"); written.append(str(p))
    if fmt in ("joins", "all"):
        p = WORKDIR / "join_candidates.csv"; p.write_text(_to_join_csv(c), encoding="utf-8"); written.append(str(p))
    log.info("catalog export -> %s", written)
    return written


log.info("Catalog tools ready: catalog_set_table, catalog_add_join (+ summary/export).")

## Model, table-type router, and the per-table profiler subgraph

`ChatVertexAI` for Gemini 2.5 (pro for synthesis, flash for the high-volume profiling). The
**router** is v2's problem-router re-pointed: a structured call that labels each table's role and
sets a cost-aware profiling depth. The **profiler** is the house ReAct subgraph (agent ⇄ tools),
invoked once per table with a fresh message list so context stays bounded.

In [ ]:
"""
Model (Gemini 2.5 via Vertex AI) + the table-type ROUTER + the per-table profiler subgraph.

- `llm`      : gemini-2.5-pro for synthesis (the overview memo).
- `llm_fast` : gemini-2.5-flash for the high-volume work (classification + per-table profiling).
- classify_table : v2's problem-router, re-pointed -- a STRUCTURED call (with_structured_output)
  that labels a table's role and maps it to a cost-aware PROFILING DEPTH (BigQuery bytes).
- profiler_app : a classic ReAct subgraph (agent <-> tools). The orchestrator invokes it once per
  table with a FRESH message list, so each table is mapped in an isolated, bounded context.
"""
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel, Field

llm = ChatVertexAI(model=MODEL, project=PROJECT or None, location=LOCATION,
                   temperature=0.1, thinking_budget=THINKING_BUDGET)
llm_fast = ChatVertexAI(model=FAST_MODEL, project=PROJECT or None, location=LOCATION,
                        temperature=0.0, thinking_budget=0)
log.info("ChatVertexAI ready: lead=%s fast=%s", MODEL, FAST_MODEL)


# --- the router: table ROLE -> cost-aware profiling depth --------------------
class TableRole(BaseModel):
    """Structured classification of a warehouse table's role."""
    role: Literal["fact", "dimension", "bridge", "reference", "staging"] = Field(
        description="fact=events/transactions; dimension=descriptive entities; bridge=m:n link; "
                    "reference=small lookup/code table; staging=raw/ingest/temp")
    reason: str = Field(description="one short sentence of evidence from the schema")


# depth = how much DATA (bytes) a table earns beyond free metadata.
DEPTH_BY_ROLE = {"fact": "profile", "dimension": "sample", "bridge": "keys",
                 "reference": "metadata", "staging": "metadata"}

CLASSIFY_SYS = (
    "Classify a BigQuery table's ROLE from its schema (column names/types/modes, row count, "
    "partitioning). fact = large, dated, many foreign keys + measures. dimension = descriptive "
    "entities keyed by an id. bridge = mostly two+ foreign keys, few attributes. reference = small "
    "lookup / code table. staging = raw/ingest/temp/_stg/_raw landing tables."
)
_classifier = llm_fast.with_structured_output(TableRole)


def classify_table(table: str, schema_json: str) -> dict:
    """Route a table to its role + a cost-aware profiling depth (v2's adaptive-compute axis)."""
    try:
        r = _classifier.invoke([SystemMessage(content=CLASSIFY_SYS),
                                HumanMessage(content=f"TABLE: {table}\nSCHEMA:\n{schema_json[:3000]}")])
        return {"role": r.role, "reason": r.reason, "depth": DEPTH_BY_ROLE.get(r.role, "sample")}
    except Exception as e:
        log.warning("classify_table failed (%s); defaulting to sample depth", e)
        return {"role": "unknown", "reason": "fallback (classification failed)", "depth": "sample"}


# --- the per-table profiler subgraph (ReAct, isolated context) ---------------
PROFILER_TOOLS = [bq_table_schema, bq_sample, bq_query, bq_columns, bq_join_overlap,
                  catalog_set_table, catalog_add_join]

PROFILER_SYSTEM = (
    "You are profiling ONE BigQuery table for a data catalog. Tools: bq_table_schema (FREE -- the "
    "schema is also given to you), bq_sample, bq_query (read-only, byte-capped), bq_columns (find a "
    "column elsewhere), bq_join_overlap, and catalog_set_table / catalog_add_join to RECORD "
    "findings. Infer the table's PURPOSE and GRAIN (what one row is), its KEY columns, and the "
    "BUSINESS RULES you can actually observe. Propose JOIN candidates via bq_columns and confirm the "
    "strongest with bq_join_overlap (record basis=value-overlap with the measured coverage and "
    "confidence=high only when overlap is strong). Record everything with catalog_set_table / "
    "catalog_add_join, citing evidence. Respect the requested DEPTH: 'metadata'=schema only; "
    "'sample'=+a few rows; 'profile'=+a small aggregate profile; 'keys'=spend bytes on FK overlap. "
    "Never invent a purpose, rule, or join. When done, STOP calling tools and reply with a "
    "one-paragraph summary."
)

OVERVIEW_SYS = (
    "You are writing a concise data-warehouse overview memo from a catalog summary: the main subject "
    "areas, the hub tables, the principal join keys, and notable business rules. Be specific and "
    "factual; do not invent anything beyond the summary."
)


def make_agent_graph(system_text: str, tools: list, model=None):
    """Compile a reusable agent<->tools ReAct graph (the house pattern from capital_allocation)."""
    sys = SystemMessage(content=system_text)
    bound = (model or llm_fast).bind_tools(tools)

    def agent_node(state: MessagesState) -> dict:
        return {"messages": [bound.invoke([sys] + state["messages"])]}

    b = StateGraph(MessagesState)
    b.add_node("agent", agent_node)
    b.add_node("tools", ToolNode(tools))
    b.add_edge(START, "agent")
    b.add_conditional_edges("agent", tools_condition)   # -> "tools" or END
    b.add_edge("tools", "agent")
    return b.compile()


profiler_app = make_agent_graph(PROFILER_SYSTEM, PROFILER_TOOLS, model=llm_fast)
log.info("Profiler subgraph + table-type router ready.")

## The spec layer — objective verification

`verify_table_entry` is the reliability hinge: it checks that an entry has a purpose, a grain and
key columns, and **re-runs the value-overlap query** for every high-confidence join (independent
recomputation, not the model's say-so). A failing entry is sent back to be re-profiled with the
specific failures as feedback.

In [ ]:
"""
The spec layer -- objective verification of a table's catalog entry.

v2 verified the agent's CODE with pytest; here we verify a catalog ENTRY the same spirit:
  1. required fields present (purpose, grain, >=1 key column), and
  2. every HIGH-confidence join is re-confirmed by ACTUALLY RE-RUNNING the value-overlap query
     above a coverage threshold -- independent recomputation, not the model's say-so.
A failing entry sends the orchestrator back to re-profile the table with the failures as feedback.
"""

def table_entry_complete(entry: dict):
    """Required fields for a 'done' table entry. Returns (ok, missing_list)."""
    missing = []
    if not (entry.get("purpose") or "").strip():
        missing.append("purpose")
    if not (entry.get("grain") or "").strip():
        missing.append("grain")
    if not entry.get("key_columns"):
        missing.append("key_columns")
    return (not missing, missing)


def verify_table_entry(table: str, *, reverify_joins: bool = True,
                       overlap_threshold: float = None) -> dict:
    """Objective check of a table's catalog entry; re-runs value-overlap for its high-confidence
    joins (the independent recomputation)."""
    threshold = VERIFY_OVERLAP_THRESHOLD if overlap_threshold is None else overlap_threshold
    c = _load_catalog()
    checks = []

    def add(name, ok, detail=""):
        checks.append({"name": name, "ok": bool(ok), "detail": detail})

    entry = c["tables"].get(table)
    add(f"{table}:catalogued", entry is not None, "no catalog entry")
    if entry is not None:
        ok, missing = table_entry_complete(entry)
        add(f"{table}:complete", ok, f"missing {missing}" if missing else "")

    high = [j for j in c["joins"]
            if (j["left"] == table or j["right"] == table) and j.get("confidence") == "high"]
    for j in high:
        label = f"join:{j['left']}.{j['left_col']}={j['right']}.{j['right_col']}"
        if not reverify_joins:
            add(label, True, "re-check disabled")
            continue
        stats = _join_overlap_stats(j["left"], j["left_col"], j["right"], j["right_col"])
        if stats is None:
            add(label, False, "overlap re-check failed/refused")
            continue
        cov = max(stats["left_cov"], stats["right_cov"])
        add(label, cov >= threshold, f"max coverage {cov:.0%} (threshold {threshold:.0%})")

    passed = sum(1 for x in checks if x["ok"])
    failed = [x for x in checks if not x["ok"]]
    return {"table": table, "passed": passed, "failed": len(failed),
            "all_passed": not failed, "checks": checks}


log.info("Spec layer ready: verify_table_entry (required fields + re-run join overlap).")

## The autonomous mapper graph

An explicit `StateGraph` over a per-table worklist — the v2 DAG-driven coverage idea in LangGraph
form. `pick → profile → verify` loops table by table (re-profiling on a failed verify, up to
`MAP_MAX_ATTEMPTS`); when the worklist is empty it routes to `synthesize`. A `MemorySaver`
checkpointer makes a long run resumable per `thread_id`.

In [ ]:
"""
The AUTONOMOUS mapper -- an explicit StateGraph that walks the warehouse table by table.

This is the v2 "DAG-driven coverage" idea in LangGraph form: the unit of work is ONE TABLE, and
coverage is a graph walk over a worklist, not a hope that the model keeps going.

        START -> pick --(table?)--> profile -> verify --(pass)--> pick
                  |  (none)                       |  (fail, retries left)
                  v                               +-----------> profile
              synthesize -> END

  pick       : pop the next table off `todo` into `current` (reset attempts).
  profile    : read the schema (free), classify_table -> role + depth, then run the bounded
               profiler subgraph on a FRESH message list (isolated context) to catalog it.
  verify     : the spec layer (verify_table_entry). pass -> done; fail w/ retries -> re-profile;
               fail w/o retries -> failed.
  synthesize : export the catalog and write a reviewed warehouse overview.

A MemorySaver checkpointer keys state by thread_id, so a long run is resumable.
"""
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


class MapperState(TypedDict):
    todo: List[str]
    done: List[str]
    failed: List[str]
    current: Optional[str]
    attempts: int
    max_attempts: int
    last_verify: dict
    overview: str


def _pick(state: MapperState) -> dict:
    todo = state["todo"]
    if todo:
        return {"current": todo[0], "todo": todo[1:], "attempts": 0}
    return {"current": None}


def _profile(state: MapperState) -> dict:
    table = state["current"]
    try:
        schema = _table_schema(table)
    except Exception as e:
        schema = json.dumps({"error": str(e)})
    route = classify_table(table, schema)
    log.info("[profile] %s role=%s depth=%s (attempt %d)",
             table, route["role"], route["depth"], state["attempts"] + 1)
    feedback = ""
    lv = state.get("last_verify") or {}
    if lv.get("checks"):
        fails = [c["name"] + ": " + c["detail"] for c in lv["checks"] if not c["ok"]]
        if fails:
            feedback = "\n\nYour previous pass FAILED these checks -- fix them:\n" + "\n".join(fails)
    task = (
        f"Catalog the BigQuery table `{table}` at DEPTH='{route['depth']}' "
        f"(inferred role={route['role']}: {route['reason']}).\n\nSCHEMA (free metadata):\n{schema}\n\n"
        "Confirm the grain (what ONE row is) and key/identifier columns; record observable business "
        "rules; use bq_columns to find the same id elsewhere and propose joins; confirm the strongest "
        "with bq_join_overlap (record basis=value-overlap + measured coverage, confidence=high only "
        "when overlap is strong). Record with catalog_set_table / catalog_add_join." + feedback)
    try:
        profiler_app.invoke({"messages": [HumanMessage(content=task)]},
                            {"recursion_limit": PROFILER_MAX_STEPS})
    except Exception as e:
        log.warning("[profile] profiler error on %s: %s", table, e)
    return {}


def _verify(state: MapperState) -> dict:
    table = state["current"]
    v = verify_table_entry(table)
    if v["all_passed"]:
        log.info("[verify] %s OK (%d checks)", table, v["passed"])
        return {"done": state["done"] + [table], "current": None, "last_verify": v}
    fails = [c["name"] for c in v["checks"] if not c["ok"]]
    if state["attempts"] + 1 < state["max_attempts"]:
        log.info("[verify] %s -> retry %s", table, fails)
        return {"attempts": state["attempts"] + 1, "last_verify": v}
    log.info("[verify] %s -> FAILED after %d attempt(s) %s", table, state["attempts"] + 1, fails)
    return {"failed": state["failed"] + [table], "current": None, "last_verify": v}


def _synthesize(state: MapperState) -> dict:
    run_catalog_export("all")
    summary = run_catalog_summary()
    try:
        overview = llm.invoke([SystemMessage(content=OVERVIEW_SYS),
                               HumanMessage(content="CATALOG SUMMARY:\n" + summary)]).content
    except Exception as e:
        overview = f"(overview generation skipped: {e})\n\n{summary}"
    try:
        (WORKDIR / "WAREHOUSE_OVERVIEW.md").write_text(overview, encoding="utf-8")
    except OSError:
        pass
    log.info("[synthesize] catalog exported; overview written")
    return {"overview": overview}


def _after_pick(state: MapperState) -> str:
    return "profile" if state["current"] else "synthesize"


def _after_verify(state: MapperState) -> str:
    return "profile" if state["current"] else "pick"


_b = StateGraph(MapperState)
_b.add_node("pick", _pick)
_b.add_node("profile", _profile)
_b.add_node("verify", _verify)
_b.add_node("synthesize", _synthesize)
_b.add_edge(START, "pick")
_b.add_conditional_edges("pick", _after_pick, {"profile": "profile", "synthesize": "synthesize"})
_b.add_edge("profile", "verify")
_b.add_conditional_edges("verify", _after_verify, {"profile": "profile", "pick": "pick"})
_b.add_edge("synthesize", END)

mapper_graph = _b.compile(checkpointer=MemorySaver())
log.info("Autonomous mapper graph compiled (pick -> profile -> verify -> synthesize).")
# Optional: mapper_graph.get_graph().print_ascii()

## Drive it

`map_warehouse()` enumerates the in-scope tables (free metadata), seeds the worklist, and runs the
graph to exhaustion. The cell below is **guarded**: it runs only when `GOOGLE_CLOUD_PROJECT` is set
and BigQuery is reachable (no synthetic data). Bound a demo with `CATALOG_MAX_TABLES`; re-run with
the same `thread_id` to resume against the on-disk catalog.

In [ ]:
"""
Driver -- enumerate the warehouse (free metadata), then run the autonomous mapper.

Guarded on REAL connectivity (no synthetic data): the run starts only if BigQuery is reachable;
otherwise it prints what to fix and skips. Set CATALOG_MAX_TABLES to bound a demo run; unset (or
pass max_tables=None) to map the whole warehouse. The run is resumable -- re-run with the same
thread_id and it continues against the on-disk catalog.
"""

def enumerate_tables(max_tables: int = None) -> List[str]:
    """All 'dataset.table' identifiers in scope, from FREE metadata only."""
    out: List[str] = []
    for ds in _list_datasets():
        try:
            out += [f"{ds}.{t}" for t in _list_tables(ds)]
        except Exception as e:
            log.warning("could not list %s: %s", ds, e)
    out.sort()
    return out[:max_tables] if max_tables else out


def map_warehouse(max_tables: int = None, thread_id: str = "map-1", fresh: bool = True) -> dict:
    """Autonomously map the warehouse: seed a per-table worklist and let the graph walk it."""
    if fresh:
        new_catalog()
    todo = enumerate_tables(max_tables)
    log.info("[map_warehouse] %d table(s) in scope", len(todo))
    init: MapperState = {"todo": todo, "done": [], "failed": [], "current": None,
                         "attempts": 0, "max_attempts": MAP_MAX_ATTEMPTS,
                         "last_verify": {}, "overview": ""}
    cfg = {"configurable": {"thread_id": thread_id},
           "recursion_limit": max(50, 6 * len(todo) + 30)}
    final = mapper_graph.invoke(init, cfg)
    return final


def _bigquery_ready() -> bool:
    try:
        _list_datasets()
        return True
    except Exception as e:
        print("BigQuery not reachable:", e)
        return False


if not PROJECT:
    print(">>> Set GOOGLE_CLOUD_PROJECT (and authenticate via "
          "`gcloud auth application-default login`), then re-run this cell.")
elif not _bigquery_ready():
    print(">>> BigQuery not ready. Check auth / GOOGLE_CLOUD_PROJECT / BQ_DATASETS, then re-run.")
else:
    cap = int(os.environ.get("CATALOG_MAX_TABLES", "12"))   # unset -> map everything
    print(f"Autonomously mapping up to {cap} table(s)...")
    print("=" * 70)
    result = map_warehouse(max_tables=cap, thread_id="catalog-demo", fresh=True)
    print("=" * 70)
    print(f"mapped: {len(result['done'])}  failed: {len(result['failed'])}  "
          f"in scope: {len(result['done']) + len(result['failed']) + len(result['todo'])}")
    print("done:  ", result["done"])
    if result["failed"]:
        print("failed:", result["failed"])
    print("\nArtifacts in:", WORKDIR)
    for fn in ("data_catalog.json", "data_dictionary.md", "join_candidates.csv",
               "WAREHOUSE_OVERVIEW.md"):
        p = WORKDIR / fn
        print(f"  {'OK ' if p.exists() else '-- '} {fn}"
              + (f"  ({p.stat().st_size} B)" if p.exists() else ""))
    if result.get("overview"):
        print("\n--- WAREHOUSE_OVERVIEW.md ---\n" + result["overview"])